# ProcessDataFrame: Frictionless Process Behavior Analysis

This notebook demonstrates the new `ProcessDataFrame` API that makes process behavior analysis intuitive and discoverable.

## Key Features

1. **Auto-completion** for column names via `data.columns.ColumnName`
2. **SDS-driven execution** - the data dictates what analyses are supported
3. **Automatic IMR charts** for simple series (SDS 0)
4. **Clear explanations** of what's running and why


In [5]:
import sys
print(sys.executable)
print(sys.path)

/Users/nicholas/Documents/projects/processbehavior/venv/bin/python
['/Users/nicholas/Documents/projects/processbehavior/examples', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/usr/local/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/nicholas/Documents/projects/processbehavior/venv/lib/python3.13/site-packages']


In [1]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame


# Set random seed for reproducibility
np.random.seed(42)

## Example 1: Simple Series (SDS 0) → IMR Chart

When you have a simple time series with no grouping structure, ProcessDataFrame automatically runs an **Individual Moving Range (IMR)** chart - just like qcc!

In [2]:
# Create simple measurement series
simple_data = pd.DataFrame({
    'Measurement': np.random.normal(100, 2, 30),
    'Time': pd.date_range('2024-01-01', periods=30, freq='D')
})

# Wrap in ProcessDataFrame
data = ProcessDataFrame(simple_data)

# Auto-completion magic! Type `data.columns.` and see your columns appear
analysis = data.analyze(
    response_var=data.columns.Measurement,
    time_var=data.columns.Time
)

# Prints explanation of SDS detection and chosen analysis


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 0: Simple Series
   Individual measurements with no rational subgrouping or time structure
   Replication: None

📈 Available charts: Imr
   Selected: IMR Chart (Individual Moving Range) (recommended)

⚠️  Not available for this SDS:
   • Xbar (requires rational subgroups)
   • S (requires rational subgroups)
   • R (requires rational subgroups)

📋 Data Configuration:
   Response: Measurement
   Time: Time

✨ Analysis Capabilities:
   • VAS residuals: Not available
   • Main effects: No
   • Interactions: No




## Example 2: Manufacturing Data with Operators (Grouped) → Xbar/S Charts

When you have rational subgroups with **replication** (multiple measurements per cell), ProcessDataFrame detects the structure and automatically runs **Xbar and S charts** to track both location (mean) and variation.

**Key**: We create a structured design where each Operator/Machine/Time combination has multiple measurements (n=2). This creates proper subgroups for Xbar/S analysis.

# Create manufacturing data with operators and machines
# Structure: Each operator works on each machine at different times
# This creates rational subgroups (multiple measurements per cell)
n_time_points = 20
n_reps = 2  # 2 measurements per operator/machine/time combination

manufacturing_data = pd.DataFrame({
    'Height': np.random.normal(50, 3, n_time_points * 3 * 3 * n_reps),  # 3 operators × 3 machines × 20 times × 2 reps
    'Operator': np.repeat(['Alice', 'Bob', 'Charlie'], n_time_points * 3 * n_reps),
    'Machine': np.tile(np.repeat(['M1', 'M2', 'M3'], n_time_points * n_reps), 3),
    'ProductionTime': np.tile(np.repeat(pd.date_range('2024-01-01', periods=n_time_points, freq='h'), n_reps), 9)
})

data = ProcessDataFrame(manufacturing_data)

# Auto-completion works here too!
analysis = data.analyze(
    response_var=data.columns.Height,
    time_var=data.columns.ProductionTime,
    grouping_vars=[data.columns.Operator, data.columns.Machine]
)

# Notice: System explains it detected SDS 1 (full replication) and why it's running Xbar/S

In [6]:
# Create manufacturing data with operators and machines
n_obs = 120
manufacturing_data = pd.DataFrame({
    'Height': np.random.normal(50, 3, n_obs),
    'Width': np.random.normal(25, 1, n_obs),
    'Operator': np.random.choice(['Alice', 'Bob', 'Charlie'], n_obs),
    'Machine': np.random.choice(['M1', 'M2', 'M3'], n_obs),
    'ProductionTime': pd.date_range('2024-01-01', periods=n_obs, freq='H')
})

data = ProcessDataFrame(manufacturing_data)

# Auto-completion works here too!
analysis = data.analyze(
    response_var=data.columns.Height,
    time_var=data.columns.ProductionTime,
    grouping_vars=[data.columns.Operator, data.columns.Machine]
)

# Notice: System explains it detected SDS 1 or 2 and why it's running Xbar/S

/var/folders/4t/g6hr0jdd4nl7zs8d2c0s9s940000gn/T/ipykernel_54159/484532436.py:8: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

SDS 6 detected: Unstructured/irregular grid.
Analysis results may be unreliable due to incomplete data coverage.
Check for missing data or irregular sampling patterns.



PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 6: Incomplete/Irregular Grid
   Sparse factor × time grid with many missing cells
   Replication: None

📈 Available charts: Imr
   Selected: IMR Chart (Individual Moving Range) (recommended)

⚠️  Not available for this SDS:
   • Xbar (requires complete grid)
   • S (requires complete grid)

📋 Data Configuration:
   Response: Height
   Time: ProductionTime
   Grouping: Operator, Machine

✨ Analysis Capabilities:
   • VAS residuals: Not available
   • Main effects: No
   • Interactions: No




In [11]:
# Get both Xbar and S chart results
result = analysis.calculate()

print("\nXbar Chart (Subgroup Means):")
print(result.dataset)

print("\nS Chart (Subgroup Variation):")



Xbar Chart (Subgroup Means):
         ProductionTime         rsg   n     Height Operator Machine  obs_id  \
5   2024-01-01 05:00:00    Alice_M1  21  46.337469    Alice      M1       0   
16  2024-01-01 16:00:00    Alice_M1  21  48.618084    Alice      M1       1   
21  2024-01-01 21:00:00    Alice_M1  21  48.844753    Alice      M1       2   
22  2024-01-01 22:00:00    Alice_M1  21  47.969234    Alice      M1       3   
47  2024-01-02 23:00:00    Alice_M1  21  49.102978    Alice      M1       4   
..                  ...         ...  ..        ...      ...     ...     ...   
87  2024-01-04 15:00:00  Charlie_M3  14  46.493966  Charlie      M3     115   
102 2024-01-05 06:00:00  Charlie_M3  14  46.813089  Charlie      M3     116   
111 2024-01-05 15:00:00  Charlie_M3  14  53.921428  Charlie      M3     117   
115 2024-01-05 19:00:00  Charlie_M3  14  52.345469  Charlie      M3     118   
117 2024-01-05 21:00:00  Charlie_M3  14  46.038630  Charlie      M3     119   

           rsg_key   

## Example 3: Quality Data with Multiple Factors

ProcessDataFrame handles complex multi-factor designs and explains the detected Sampling Design State.

In [12]:
# Create data with 2 factors and time
factors = pd.DataFrame({
    'Factor1': ['Low', 'High'] * 60,
    'Factor2': ['A', 'B', 'C', 'D'] * 30
})

quality_data = pd.DataFrame({
    'Strength': np.random.normal(100, 5, 120),
    'Temperature': np.random.choice(['Low', 'High'], 120),
    'Pressure': np.random.choice(['A', 'B'], 120),
    'Batch': range(1, 121)
})

data = ProcessDataFrame(quality_data)

analysis = data.analyze(
    response_var=data.columns.Strength,
    time_var=data.columns.Batch,
    grouping_vars=[data.columns.Temperature, data.columns.Pressure]
)

# System detects SDS and explains capabilities

SDS 6 detected: Unstructured/irregular grid.
Analysis results may be unreliable due to incomplete data coverage.
Check for missing data or irregular sampling patterns.



PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 6: Incomplete/Irregular Grid
   Sparse factor × time grid with many missing cells
   Replication: None

📈 Available charts: Imr
   Selected: IMR Chart (Individual Moving Range) (recommended)

⚠️  Not available for this SDS:
   • Xbar (requires complete grid)
   • S (requires complete grid)

📋 Data Configuration:
   Response: Strength
   Time: Batch
   Grouping: Temperature, Pressure

✨ Analysis Capabilities:
   • VAS residuals: Not available
   • Main effects: No
   • Interactions: No




## Example 4: Zero-Centered Analysis

Sometimes you want to center your data at zero to focus on deviations. ProcessDataFrame makes this easy.

In [13]:
# Data that's not centered at zero
offset_data = pd.DataFrame({
    'Value': np.random.normal(1000, 10, 50),
    'Sequence': range(1, 51)
})

data = ProcessDataFrame(offset_data)

analysis = data.analyze(
    response_var=data.columns.Value,
    time_var=data.columns.Sequence,
    zero_center=True  # Subtract mean to focus on variation
)

result = analysis.calculate()
print("\nZero-centered analysis:")
print(result)


PROCESS BEHAVIOR ANALYSIS

📊 Detected SDS 0: Simple Series
   Individual measurements with no rational subgrouping or time structure
   Replication: None

📈 Available charts: Imr
   Selected: IMR Chart (Individual Moving Range) (recommended)

⚠️  Not available for this SDS:
   • Xbar (requires rational subgroups)
   • S (requires rational subgroups)
   • R (requires rational subgroups)

📋 Data Configuration:
   Response: Value
   Time: Sequence

✨ Analysis Capabilities:
   • VAS residuals: Not available
   • Main effects: No
   • Interactions: No



Zero-centered analysis:
ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 0
Description: No grouping or time structure

Analysis Type: Imr
Response Variable: Value
Time Variable: Sequence

Observations: 50
Charts: all

Capabilities:
  Residuals: ✗
  Effects: ✗
  Interactions: ✗


## Why This API is Better

### Before (Old API)
```python
# Easy to make typos in column names!
spec = {
    'analysis_type': 'Imr',  # User must know which type
    'response_var': 'Measurment',  # Typo! Will fail
    'time_var': 'Time',
    'rsg_vars': None
}
analysis = Analysis(df, spec)
```

### After (New API)
```python
# Auto-completion prevents typos!
# System chooses correct analysis type!
data = ProcessDataFrame(df)
analysis = data.analyze(
    response_var=data.columns.Measurement,  # IDE autocompletes
    time_var=data.columns.Time
)
# Prints: "Detected SDS 0: Running IMR Chart (Individual Moving Range)"
```

## Benefits

1. **No more typos** - IDE autocomplete ensures valid column names
2. **No more wrong analysis types** - System detects SDS and chooses appropriately
3. **Transparency** - Always explains what it's doing and why
4. **Follows the data** - Analysis adapts to your data structure
5. **Pythonic** - Clean, readable, discoverable API


## Understanding Sampling Design States (SDS)

ProcessDataFrame automatically detects which of 7 SDS categories your data falls into:

- **SDS 0**: No grouping or time → IMR chart
- **SDS 1**: Full replication (all cells n≥2) → Full VAS + Xbar/S
- **SDS 2**: No replication (all cells n=1) → Residuals + Xbar/S
- **SDS 3**: Partial replication (mixed) → Hybrid approach
- **SDS 4**: Single condition over time → Time series analysis
- **SDS 5**: Nested design → Variance components
- **SDS 6**: Irregular grid → Adaptive limits

You don't need to know these - the system figures it out and tells you!